# Microsoft Agent Framework — Azure OpenAI (Responses API)

I dette kodeeksemplet skal du bruke **Microsoft Agent Framework (MAF)** for å lage en enkel agent som støttes av **Azure OpenAI** ved bruk av **Responses API**.

> **Migrasjonsmerknad:** Dette eksemplet brukte tidligere Semantic Kernel med GitHub Models. Det har blitt migrert til Microsoft Agent Framework, og GitHub Models (utdatert, avsluttes i juli 2026) er erstattet med Azure OpenAI, som støtter Responses API. `OpenAIChatClient` i MAF retter seg mot Azure OpenAI sitt stabile `/openai/v1/` endepunkt og bruker som standard Responses API.

Hensikten med dette eksemplet er å demonstrere trinnene som senere vil bli brukt i flere kodeeksempler ved implementering av ulike agentmønstre.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Importer de nødvendige Python-pakkene


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definere et verktøy

I Microsoft Agent Framework er et **verktøy** en enkel Python-funksjon dekorert med `@tool` som agenten kan kalle. Nedenfor definerer vi et verktøy som returnerer et tilfeldig feriemål og unngår å gjenta det forrige.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Lage agenten

Her lager vi agenten kalt `TravelAgent`.

I dette eksempelet bruker vi veldig grunnleggende instruksjoner. Du kan gjerne endre disse instruksjonene for å se hvordan agentens oppførsel endres.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Kjøre agenten

Nå kan vi kjøre agenten. Vi oppretter en `AgentSession` slik at agenten husker samtalen over flere runder, og sender deretter to `user_inputs`. Den første ber om en tur; den andre sier at brukeren ikke likte forslaget og ber om et annet — agenten bruker sesjonshistorikken pluss verktøyet `get_random_destination` for å svare.

Du kan endre disse meldingene for å se hvordan agenten reagerer forskjellig. Svarene **strømmes** token for token.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
